<a href="https://colab.research.google.com/github/syankov-ai/Medium/blob/main/event_calendar/Event_Calendar_Streamlit_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Packages

In [1]:
!pip install streamlit -q

In [2]:
!pip install streamlit-calendar -q

# Create the App

In [3]:
%%writefile app.py
import streamlit as st
import json
from datetime import date, datetime, timezone
from dataclasses import dataclass
from typing import List, Optional
from collections import defaultdict

@dataclass
class Event:
    id: str
    title: str
    date: date        # ← only date, no time
    description: str
    location: Optional[str] = None
    tags: Optional[List[str]] = None
    status: Optional[str] = None

def load_events_from_json(path: str) -> List[Event]:
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    events = []
    for item in raw:
        # Assume datetime in JSON is YYYY-MM-DD string
        d = datetime.strptime(item["datetime"], "%Y-%m-%d").date()
        events.append(
            Event(
                id=item["id"],
                title=item["title"],
                date=d,
                description=item.get("description", ""),
                location=item.get("location"),
                tags=item.get("tags", []),
                status=item.get("status"),
            )
        )
    return events

def save_events_to_json(path: str, events: List[Event]) -> None:
    serializable = []
    for e in events:
        serializable.append(
            {
                "id": e.id,
                "title": e.title,
                "datetime": e.date.strftime("%Y-%m-%d"),   # store as date string
                "description": e.description,
                "location": e.location,
                "tags": e.tags,
                "status": e.status,
            }
        )
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serializable, f, ensure_ascii=False, indent=2)

def event_to_post_title(event: Event) -> str:
    date_str = event.date.strftime("%d %b %Y")
    return f"{event.title} – {date_str}"

def event_to_post_body(event: Event) -> str:
    date_str = event.date.strftime("%A, %d %B %Y")
    tags_str = ", ".join(event.tags or [])
    body = f"""
<h2>{event.title}</h2>
<p><strong>Date:</strong> {date_str}</p>
"""
    if event.location:
        body += f"<p><strong>Location:</strong> {event.location}</p>\n"
    body += f"""
<p><strong>Details:</strong></p>
<p>{event.description}</p>
"""
    if tags_str:
        body += f"<p><strong>Tags:</strong> {tags_str}</p>\n"
    body += """
<hr/>
<p>This post was generated automatically from my event calendar.</p>
"""
    return body.strip()

def should_post_event(event: Event, now: Optional[datetime] = None) -> bool:
    if now is None:
        now = datetime.now(timezone.utc).replace(tzinfo=None)
    if event.status == "posted":
        return False
    # Consider event “ready” when its date is <= today
    return event.date <= now.date()

def select_events_to_post(events: List[Event]) -> List[Event]:
    return [e for e in events if should_post_event(e)]

# Group events by date for daily view
def group_events_by_date(events: List[Event]) -> dict:
    groups = defaultdict(list)
    for e in events:
        date_key = e.date.isoformat()
        groups[date_key].append(e)
    return dict(sorted(groups.items()))

st.set_page_config(page_title="Event Calendar Bot", layout="wide")

st.title("📅 Event Calendar Bot")
st.markdown("View your events grouped by day, preview posts, and prepare for social media publishing.")

json_path = "events.json"

st.sidebar.header("Events")
events = None
try:
    events = load_events_from_json(json_path)
    st.sidebar.success(f"Loaded {len(events)} events from events.json")
except FileNotFoundError:
    st.sidebar.warning("Create events.json first!")
    st.sidebar.info("Use the sample below or your own data.")

if not events:
    st.info("""
    **Quick start:** Create `events.json` with this sample:
    ```json
    [
      {
        "id": "event-001",
        "title": "Baba Marta Day",
        "datetime": "2026-03-01",
        "description": "Spring tradition celebration",
        "tags": ["tradition", "spring"],
        "status": "pending"
      }
    ]
    ```
    """)

if events:
    # Daily view
    st.subheader("📅 Daily Events Overview")
    daily_groups = group_events_by_date(events)

    for date_str, day_events in daily_groups.items():
        with st.expander(f"{date_str} ({len(day_events)} events)"):
            for event in day_events:
                st.markdown(f"**{event.title}**")
                if event.description:
                    st.write(event.description[:100] + "...")
                st.caption(f"Status: {event.status or 'pending'} | Tags: {', '.join(event.tags or [])}")

    # Ready to post
    to_post = select_events_to_post(events)
    if to_post:
        st.subheader(f"🚀 Ready to Post ({len(to_post)} events)")
        for event in to_post:
            with st.expander(event_to_post_title(event)):
                st.markdown("**Preview:**")
                st.markdown(event_to_post_body(event), unsafe_allow_html=True)

                col_a, col_b, col_c = st.columns([2, 2, 1])
                with col_a:
                    if st.button("📱 Post to Social Media", key=f"social_{event.id}"):
                        st.success(f"✅ Social post prepared for '{event.title}' (integration coming soon)")
                with col_b:
                    if st.button("✏️ Mark as Posted", key=f"mark_{event.id}"):
                        event.status = "posted"
                        save_events_to_json(json_path, events)
                        st.success("Status updated!")
                        st.rerun()
                with col_c:
                    st.caption("Future: 15‑min staggered publish")
    else:
        st.info("No events ready to post.")


Overwriting app.py


# Making Streamlit accessible with `ngrok`

Since the Streamlit app isn't reachable directly, we'll use `ngrok` to create a publicly accessible URL. This involves:
1. Installing the `pyngrok` library.
2. Obtaining an `ngrok` authtoken from their website (you'll need a free account).
3. Using the authtoken to authorize `ngrok`.
4. Running Streamlit through `ngrok` to get a public URL.

First, let's install `pyngrok`.

In [4]:
!pip install pyngrok -q

Now, you'll need to set up your `ngrok` authtoken.
1. Go to [ngrok.com](https://ngrok.com/) and sign up for a free account.
2. On your dashboard, you'll find your authtoken.
3. In Colab, go to the left-hand panel, click on the "🔑" (Secrets) icon.
4. Add a new secret named `NGROK_AUTH_TOKEN` and paste your authtoken as the value.
5. Make sure "Notebook access" is enabled for this secret.

# Run the Application

In [7]:
from pyngrok import ngrok
from google.colab import userdata

# Terminate any previous ngrok tunnels
ngrok.kill()

# Get the authtoken from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authtoken set.")
else:
    print("NGROK_AUTH_TOKEN not found in Colab secrets. Please set it.")

# Start a ngrok tunnel for Streamlit (which runs on port 8501 by default)
public_url = ngrok.connect(8501)
print(f"Streamlit Public URL: {public_url}")

# Now, run your Streamlit app
# This command will block, so the Streamlit app will be running as long as this cell is executing.
!streamlit run app.py --server.port 8501

ngrok authtoken set.
Streamlit Public URL: NgrokTunnel: "https://unpricked-lorriane-nonestimably.ngrok-free.dev" -> "http://localhost:8501"





  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.245.203.48:8501

  Stopping...
